##### collapse:

In [8]:
from torchvision import transforms, datasets
import torch

transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root="data",
    download=False,
    transform=transform,
    train=True
)

test_data = datasets.MNIST(
    root="data",
    download=False,
    transform=transform,
    train=False
)

In [9]:
from torch.utils.data import DataLoader
import torch.nn as nn

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

### Problem with the current CNN

In [10]:
old_cnn_model = nn.Sequential(
    nn.Conv2d(1, 8, 3),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(8 * 13 * 13, 10)
)

Issues
- only 1 convolution layer
- only 8 filters
- no padding → spatial info lost early
- pooling happens too early
- almost no feature hierarchy

Conclusion
- this CNN is barely stronger than a linear model

So we strengthen it step by step.

### Step 1 – Add padding

Old

`nn.Conv2d(1, 8, 3)`

New

`nn.Conv2d(1, 8, kernel_size=3, padding=1)`

#### What padding=1 does

- adds a 1-pixel border around the image
- prevents spatial shrinking

Effect on size
- before: 28 → 26
- after: 28 → 28

Why this matters
- spatial position information is preserved longer
- CNNs need this for robustness

### Step 2 – Add depth (second convolution layer)

Why depth exists
- first conv layer → edges
- second conv layer → combinations of edges (shapes)

New layer

`nn.Conv2d(8, 16, kernel_size=3, padding=1)`


Breakdown
- input channels = 8 (from previous layer)
- output channels = 16 (richer patterns)
- same 3×3 local view
- padding preserves spatial size

This creates hierarchical feature learning.

### Step 3 – Delay pooling

Pooling
- reduces spatial resolution
- throws away position information

So
- we do NOT pool immediately
- we pool after learning structure

### The new CNN model

In [11]:
new_cnn_model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),   # [1,28,28] → [8,28,28]
    nn.ReLU(),

    nn.Conv2d(8, 16, kernel_size=3, padding=1),  # [8,28,28] → [16,28,28]
    nn.ReLU(),

    nn.MaxPool2d(2),                             # [16,28,28] → [16,14,14]

    nn.Flatten(),                                # → [16*14*14 = 3136]
    nn.Linear(16 * 14 * 14, 10)
)

- 1 → grayscale input
- 8 → small set of low-level patterns
- 16 → richer mid-level patterns
- 3×3 → standard local receptive field
- padding=1 → preserve spatial layout
- 14×14 → pooling halves 28
- 3136 → 16 × 14 × 14
- 10 → digit classes

### What changed

Before
- CNN ≈ linear model with locality

Now
- CNN builds hierarchical spatial features
- spatial layout survives longer
- robustness can emerge

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
loss_fn = nn.CrossEntropyLoss()
epochs = 3

def train_model(model):
    print("Training: ")
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        
        for images, labels in train_loader:
            model.to(device)
            images = images.to(device)
            labels = labels.to(device)
    
            preds = model(images)
            loss = loss_fn(preds, labels)
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
        print(f"Epoch: {epoch} | Loss: {loss.item():.4f}")

In [ ]:
def eval_model(model):
    print("Testing: ")
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            preds = model(images)
            predicted = preds.argmax(dim=1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print("Accuracy: ", accuracy)


In [25]:
print("Old CNN")
train_model(old_cnn_model)
eval_model(old_cnn_model)
print("\n")

print("New CNN")
train_model(new_cnn_model)
eval_model(new_cnn_model)

Old CNN
Training
Epoch: 0 | Loss: 0.0390
Epoch: 1 | Loss: 0.0166
Epoch: 2 | Loss: 0.0124
Testing: 
0.9778


New CNN
Training
Epoch: 0 | Loss: 0.0082
Epoch: 1 | Loss: 0.0074
Epoch: 2 | Loss: 0.0008
Testing: 
0.9862
